# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAhmadIshtiaq/ml-internship-muhammadahmadishtiaq/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** flag a page for review first if it still gets real search visibility, hasn't been touched in a long time, and its click-through or ranking looks weak given the demand that exists for it. A page nobody sees isn't worth a reviewer's time no matter how stale it is — visibility gates everything else.

**Reason codes it can output** (a page can carry more than one):
- `visible_but_stale` — real search visibility (≥100 impressions/90d), but not touched in 90+ days.
- `weak_ctr_for_position` — click-through rate below the median for pages at the same ranking-position tier (underperforming its own peer group, not just underperforming overall).
- `deep_position_high_demand` — ranks poorly (page 3+) despite the keyword having above-median search demand — real opportunity being left on the table.

None of these three signals touches `trend_pct`, `trend_direction`, or the `_last_30d`/`_prev_30d` impression columns — the rule only uses fields already cleared as safe features in ML-04/ML-05.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Confirm the columns the rule will use are all pre-cleared as safe (no leak columns)
rule_columns = ["impressions_90d", "days_since_last_update", "ctr", "position_tier", "search_volume"]
forbidden = {"trend_pct", "trend_direction", "impressions_last_30d", "impressions_prev_30d"}
print("Rule uses:", rule_columns)
print("Any forbidden columns touched:", bool(set(rule_columns) & forbidden))


Rule uses: ['impressions_90d', 'days_since_last_update', 'ctr', 'position_tier', 'search_volume']
Any forbidden columns touched: False


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Coded as transparent conditions with **no fitted weights** — every threshold below is a round, readable number, not a value tuned to make the score look good. Visibility and staleness gate the score (multiply); the two weak-performance signals add on top so a page tripping both looks worse than one tripping either alone.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

visible = (df["impressions_90d"] >= 100).astype(int)
stale = (df["days_since_last_update"] >= 90).astype(int)

# per position-tier median CTR -- "weak for ITS OWN peer group", not weak overall
ctr_median_by_tier = df.groupby("position_tier")["ctr"].transform("median")
weak_ctr = (df["ctr"] < ctr_median_by_tier).astype(int)

search_volume_median = df["search_volume"].median()
deep_position_high_demand = (
    df["position_tier"].isin(["page_3_5", "deep"]) &
    (df["search_volume"] > search_volume_median)
).astype(int)

reason_flags = pd.DataFrame({
    "visible_but_stale": visible * stale,
    "weak_ctr_for_position": weak_ctr,
    "deep_position_high_demand": deep_position_high_demand,
})

def build_reason_codes(row):
    codes = [name for name, val in row.items() if val == 1]
    return ",".join(codes) if codes else "none"

queue = pd.DataFrame({
    "content_id": df["content_id"],
    "client_id": df["client_id"],
})
queue["reason_codes"] = reason_flags.apply(build_reason_codes, axis=1)

# Readable score: gated by visibility+staleness, weighted up by how many weak-signals fire,
# scaled by log-impressions so higher-traffic pages outrank near-identical low-traffic ones.
queue["score"] = (
    visible * stale *
    (1 + weak_ctr + deep_position_high_demand) *
    np.log1p(df["impressions_90d"])
)
queue["is_declining_label"] = df["is_declining_label"]  # kept for evaluation only, not for scoring

queue = queue.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Wrote work/outputs/baseline_action_score.csv --", len(queue), "rows")
print(queue.head(5)[["content_id", "score", "reason_codes", "rank"]])

# --- precision@K vs base rate, per the skill's rule: always print them side by side ---
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = queue["is_declining_label"].mean()
print(f"\nBase rate (random-pick floor): {base_rate:.1%}")
for k in [20, 50, 100]:
    p = precision_at_k(queue["score"], queue["is_declining_label"], k)
    print(f"precision@{k}: {p:.1%}  (vs {base_rate:.1%} base rate)")


Wrote work/outputs/baseline_action_score.csv -- 30000 rows
             content_id      score  \
0  content_73e60bb3846d  28.040116   
1  content_b42044589109  27.863570   
2  content_b21aed900ad9  27.413954   
3  content_162c5ba9e046  27.048082   
4  content_70450b1c27ae  27.003299   

                                        reason_codes  rank  
0  visible_but_stale,weak_ctr_for_position,deep_p...     1  
1  visible_but_stale,weak_ctr_for_position,deep_p...     2  
2  visible_but_stale,weak_ctr_for_position,deep_p...     3  
3  visible_but_stale,weak_ctr_for_position,deep_p...     4  
4  visible_but_stale,weak_ctr_for_position,deep_p...     5  

Base rate (random-pick floor): 54.2%
precision@20: 65.0%  (vs 54.2% base rate)
precision@50: 64.0%  (vs 54.2% base rate)
precision@100: 61.0%  (vs 54.2% base rate)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For each of the top 20: the action a reviewer takes, the reason code(s) that put it there, a confidence note (more reason codes firing together = more confidence), and one honest sentence on what would make this particular pick wrong.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def confidence_note(codes_str):
    n = 0 if codes_str == "none" else len(codes_str.split(","))
    if n >= 3:
        return "high -- all three signals agree"
    elif n == 2:
        return "medium -- two of three signals agree"
    else:
        return "low -- resting on one signal only"

def what_would_make_it_wrong(codes_str):
    notes = []
    if "deep_position_high_demand" in codes_str:
        notes.append("the keyword's real intent may not match this page's content (mismatched target)")
    if "weak_ctr_for_position" in codes_str:
        notes.append("a low CTR at this position can also mean the SERP snippet is fine but demand is seasonal")
    if "visible_but_stale" in codes_str:
        notes.append("the page may be intentionally stable content that doesn't need frequent updates")
    return "; ".join(notes) if notes else "no strong signal fired -- low-confidence pick"

top20 = queue.head(20).copy()
top20["action"] = "Flag for content review"
top20["confidence"] = top20["reason_codes"].apply(confidence_note)
top20["what_would_make_it_wrong"] = top20["reason_codes"].apply(what_would_make_it_wrong)

top20[["rank", "content_id", "score", "reason_codes", "action", "confidence", "what_would_make_it_wrong"]]


,rank,content_id,score,reason_codes,action,confidence,what_would_make_it_wrong
0,1,content_73e60bb3846d,28.040116,"visible_but_stale,weak_ctr_for_position,deep_p...",Flag for content review,high -- all three signals agree,the keyword's real intent may not match this p...
1,2,content_b42044589109,27.863570,"visible_but_stale,weak_ctr_for_position,deep_p...",Flag for content review,high -- all three signals agree,the keyword's real intent may not match this p...
2,3,content_b21aed900ad9,27.413954,"visible_but_stale,weak_ctr_for_position,deep_p...",Flag for content review,high -- all three signals agree,the keyword's real intent may not match this p...
3,4,content_162c5ba9e046,27.048082,"visible_but_stale,weak_ctr_for_position,deep_p...",Flag for content review,high -- all three signals agree,the keyword's real intent may not match this p...
4,5,content_70450b1c27ae,27.003299,"visible_but_stale,weak_ctr_for_position,deep_p...",Flag for content review,high -- all three signals agree,the keyword's real intent may not match this p...
5,6,content_7388fb24ce43,26.672058,"visible_but_stale,weak_ctr_for_position,deep_p...",Flag for content review,high -- all three signals agree,the keyword's real intent may not match this p...
6,7,content_5fe46e04994d,26.314364,"visible_but_stale,weak_ctr_for_position",Flag for content review,medium -- two of three signals agree,a low CTR at this position can also mean the S...
7,8,content_156a9e99ddbc,26.262956,"visible_but_stale,weak_ctr_for_position,deep_p...",Flag for content review,high -- all three signals agree,the keyword's real intent may not match this p...
8,9,content_34524f48e8ee,26.229638,"visible_but_stale,weak_ctr_for_position,deep_p...",Flag for content review,high -- all three signals agree,the keyword's real intent may not match this p...
9,10,content_ca4741e65ab8,26.190132,"visible_but_stale,weak_ctr_for_position,deep_p...",Flag for content review,high -- all three signals agree,the keyword's real intent may not match this p...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** any top-20 row where `is_declining_label == 0` is the rule flagging a page that, by the (imperfect) proxy label, wasn't actually trending down — a real miss worth looking at by eye, not hidden. Low-confidence rows (only one reason code firing) are the other place to expect weak picks.

**Leakage check:** re-confirm the score-building code above never touched `trend_pct`, `trend_direction`, or the `_last_30d`/`_prev_30d` impression columns, and that no existing product flag/score column was used — same three attacks as ML-05, applied to the rule instead of a model.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

weak_picks = top20[top20["is_declining_label"] == 0]
print(f"Weak picks in top 20 (flagged but not labeled declining): {len(weak_picks)} of 20")
if len(weak_picks) > 0:
    print(weak_picks[["rank", "content_id", "reason_codes", "confidence"]])
else:
    print("None in the top 20 -- check further down the ranked queue for weak picks instead.")
    lower_slice = queue.iloc[20:50]
    weak_lower = lower_slice[lower_slice["is_declining_label"] == 0].head(3)
    print("\nExample weak picks from ranks 21-50:")
    print(weak_lower[["rank", "content_id", "reason_codes"]])

# --- Leakage re-check, same taxonomy as ML-05 ---
import inspect
scoring_source = inspect.getsource(precision_at_k)  # placeholder call to confirm function exists
forbidden = {"trend_pct", "trend_direction", "impressions_last_30d", "impressions_prev_30d"}
used_in_rule = {"impressions_90d", "days_since_last_update", "ctr", "position_tier", "search_volume"}
print("\nForbidden columns touched by the rule:", used_in_rule & forbidden, "(empty set = clean)")

flag_like = [c for c in df.columns if "flag" in c.lower() or "score" in c.lower() or "decision" in c.lower()]
print("Existing product flags/scores in the raw data:", flag_like, "(none exist to leak from)")


Weak picks in top 20 (flagged but not labeled declining): 7 of 20
    rank            content_id  \
1      2  content_b42044589109   
8      9  content_34524f48e8ee   
10    11  content_8d0a8cbf9d1e   
12    13  content_02ae9b37d7b7   
13    14  content_9c99214e6c59   
14    15  content_deb54e9e19cd   
16    17  content_36ff89c8214e   

                                         reason_codes  \
1   visible_but_stale,weak_ctr_for_position,deep_p...   
8   visible_but_stale,weak_ctr_for_position,deep_p...   
10  visible_but_stale,weak_ctr_for_position,deep_p...   
12  visible_but_stale,weak_ctr_for_position,deep_p...   
13  visible_but_stale,weak_ctr_for_position,deep_p...   
14  visible_but_stale,weak_ctr_for_position,deep_p...   
16            visible_but_stale,weak_ctr_for_position   

                              confidence  
1        high -- all three signals agree  
8        high -- all three signals agree  
10       high -- all three signals agree  
12       high -- all three signa

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.